# Week 2 · Day 3 — LoRA 模型架构搭建与验证

**模型**：`PrithviMAE`（来自 `../hls-foundation-os/pretrained_models/prithvi_100m/prithvi_mae.py`）  
**权重**：`Prithvi_100M.pt`（与 Week 1 notebook 04 完全相同的加载方式）  
**输入**：`(B, 6, 3, 224, 224)`（T=3，复制单帧）  
**LoRA 位置**：`encoder.blocks.*.attn.qkv`（Week 1 notebook 03 确认）

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Microsoft YaHei')
matplotlib.rcParams['axes.unicode_minus'] = False

import sys
import yaml
import torch
import torch.nn as nn
from pathlib import Path
from peft import LoraConfig, get_peft_model, TaskType

MODEL_DIR = Path(r'D:\geology AI\hls-foundation-os\pretrained_models\prithvi_100m')
sys.path.insert(0, str(MODEL_DIR))
from prithvi_mae import PrithviMAE

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {DEVICE}')
print(f'模型目录: {MODEL_DIR.resolve()}')
print(f'prithvi_mae.py 存在: {(MODEL_DIR / "prithvi_mae.py").exists()}')
print(f'Prithvi_100M.pt 存在: {(MODEL_DIR / "Prithvi_100M.pt").exists()}')

c:\ProgramData\miniconda3\envs\prithvi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


设备: cuda
模型目录: D:\geology AI\hls-foundation-os\pretrained_models\prithvi_100m
prithvi_mae.py 存在: True
Prithvi_100M.pt 存在: True


## 3.1 加载 PrithviMAE（与 Week 1 notebook 04 相同方式）

In [2]:
# Week 1 notebook 04 的确切加载方式
with open(MODEL_DIR / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# 权重文件：Prithvi_100M.pt，直接就是 state_dict
ckpt = torch.load(MODEL_DIR / 'Prithvi_100M.pt', map_location='cpu')
state_dict = ckpt   # ← 注意：不需要 .get('model', ckpt)，直接就是权重

# 从 state_dict 读取维度（Week 1 notebook 04 的方式）
embed_dim         = state_dict['encoder.norm.weight'].shape[0]      # 768
decoder_embed_dim = state_dict['decoder.decoder_embed.bias'].shape[0]  # 512
num_heads         = embed_dim // 64   # 12

print(f'embed_dim={embed_dim}  decoder_embed_dim={decoder_embed_dim}  num_heads={num_heads}')

prithvi = PrithviMAE(
    img_size=224, patch_size=16, num_frames=3,   # T=3，与训练时相同
    tubelet_size=1, in_chans=6,
    embed_dim=embed_dim, depth=12, num_heads=num_heads,
    decoder_embed_dim=decoder_embed_dim,
    decoder_depth=8, decoder_num_heads=16,
    mlp_ratio=4.0, norm_pix_loss=False,
)
prithvi.load_state_dict(state_dict, strict=False)
prithvi.eval()

total_p = sum(p.numel() for p in prithvi.parameters())
print(f'\nPrithviMAE 加载成功，参数量: {total_p/1e6:.1f}M')

embed_dim=768  decoder_embed_dim=512  num_heads=12

PrithviMAE 加载成功，参数量: 112.6M


## 3.2 探查 Encoder Attention 层名称

Week 1 notebook 03 已确认：LoRA 插入位置 = `encoder.blocks.*.attn.qkv`

In [3]:
print('=== encoder.blocks.0 的所有 Linear 层 ===')
for name, module in prithvi.named_modules():
    if 'encoder.blocks.0' in name and isinstance(module, nn.Linear):
        print(f'  {name:<55}  in={module.in_features:<5}  out={module.out_features}')

print('\n=== 验证 LoRA target 路径 ===')
# 检查 qkv 和 proj 是否在正确位置
for name, module in prithvi.named_modules():
    if 'encoder.blocks.0.attn' in name and isinstance(module, nn.Linear):
        print(f'  ✓ {name}  → 将被 LoRA 替换')

=== encoder.blocks.0 的所有 Linear 层 ===
  encoder.blocks.0.attn.qkv                                in=768    out=2304
  encoder.blocks.0.attn.proj                               in=768    out=768
  encoder.blocks.0.mlp.fc1                                 in=768    out=3072
  encoder.blocks.0.mlp.fc2                                 in=3072   out=768

=== 验证 LoRA target 路径 ===
  ✓ encoder.blocks.0.attn.qkv  → 将被 LoRA 替换
  ✓ encoder.blocks.0.attn.proj  → 将被 LoRA 替换


## 3.3 分割解码头

**Token 处理**（T=3 时）：
```
输入 (B,6,3,224,224) → forward_encoder → latent (B,589,768)
去 CLS → (B,588,768) → reshape (B,3,196,768)
时间维度取均值 → (B,196,768) → (B,768,14,14)
解码 → (B,1,224,224)
```

In [5]:
class SegDecoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        self.up1  = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, 2, 2), nn.BatchNorm2d(256), nn.GELU())
        self.up2  = nn.Sequential(
            nn.ConvTranspose2d(256, 64, 4, 4),        nn.BatchNorm2d(64),  nn.GELU())
        self.up3  = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, 2),         nn.BatchNorm2d(32),  nn.GELU())
        self.head = nn.Conv2d(32, 1, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.up3(self.up2(self.up1(x))))

# 验证
dec_test = SegDecoder()
out = dec_test(torch.zeros(2, 768, 14, 14))
print(f'Decoder 输出: {out.shape}  ← 期望 (2,1,224,224)')
assert out.shape == (2, 1, 224, 224); print('✓ Decoder 形状正确')

Decoder 输出: torch.Size([2, 1, 224, 224])  ← 期望 (2,1,224,224)
✓ Decoder 形状正确


In [18]:
class PrithviSegModel(nn.Module):
    NUM_FRAMES = 3
    EMBED_DIM  = 768
    GRID_SIZE  = 14

    def __init__(self, prithvi_model):
        super().__init__()
        self.prithvi = prithvi_model
        self.decoder = SegDecoder()

    def forward(self, x):
        B = x.shape[0]
        x_t = x.unsqueeze(2).repeat(1, 1, self.NUM_FRAMES, 1, 1)  # (B,6,3,224,224)

        # 取最后一层输出，去掉 CLS token
        tokens = self.prithvi.base_model.model.forward_features(x_t)[-1]
        # tokens: (B, 589, 768)
        tokens = tokens[:, 1:, :]                                   # (B, 588, 768)

        # 时间维度取均值
        tokens = tokens.reshape(
            B, self.NUM_FRAMES, self.GRID_SIZE**2, self.EMBED_DIM
        ).mean(1)                                                    # (B, 196, 768)

        feat = tokens.transpose(1, 2).reshape(
            B, self.EMBED_DIM, self.GRID_SIZE, self.GRID_SIZE
        )                                                            # (B, 768, 14, 14)

        return self.decoder(feat)                                    # (B, 1, 224, 224)


model = PrithviSegModel(prithvi).to(DEVICE)
print('PrithviSegModel 构建完成')

PrithviSegModel 构建完成


## 3.4 应用 LoRA

In [19]:
# 重新加载干净的 prithvi（避免重复应用 LoRA）
ckpt = torch.load(MODEL_DIR / 'Prithvi_100M.pt', map_location='cpu')
state_dict = ckpt

embed_dim         = state_dict['encoder.norm.weight'].shape[0]
decoder_embed_dim = state_dict['decoder.decoder_embed.bias'].shape[0]
num_heads         = embed_dim // 64

prithvi = PrithviMAE(
    img_size=224, patch_size=16, num_frames=3,
    tubelet_size=1, in_chans=6,
    embed_dim=embed_dim, depth=12, num_heads=num_heads,
    decoder_embed_dim=decoder_embed_dim,
    decoder_depth=8, decoder_num_heads=16,
    mlp_ratio=4.0, norm_pix_loss=False,
)
prithvi.load_state_dict(state_dict, strict=False)

# 构建完整模型
model = PrithviSegModel(prithvi).to(DEVICE)

# 应用 LoRA（只应用一次）
LORA_R     = 8
LORA_ALPHA = 16

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    bias='none',
    target_modules=['qkv', 'proj'],
    task_type=TaskType.FEATURE_EXTRACTION,
)
model.prithvi = get_peft_model(prithvi, lora_cfg)
model.prithvi.print_trainable_parameters()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n总参数:    {total/1e6:.2f}M')
print(f'可训练:    {trainable/1e6:.2f}M ({100*trainable/total:.1f}%)')

trainable params: 657,408 || all params: 113,296,896 || trainable%: 0.5803

总参数:    114.35M
可训练:    1.72M (1.5%)


## 3.5 前向传播验证

In [20]:
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 6, 224, 224).to(DEVICE)
    out   = model(dummy)

print(f'输入: {dummy.shape}')
print(f'输出: {out.shape}  ← 期望 (2,1,224,224)')
print(f'Logits 范围: [{out.min().item():.3f}, {out.max().item():.3f}]')
assert out.shape == (2, 1, 224, 224)
print('\n✓ 前向传播验证通过')

输入: torch.Size([2, 6, 224, 224])
输出: torch.Size([2, 1, 224, 224])  ← 期望 (2,1,224,224)
Logits 范围: [-0.185, 0.204]

✓ 前向传播验证通过


## 3.6 显存测试

In [21]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    model.train()
    x = torch.randn(4, 6, 224, 224).to(DEVICE)
    model(x).mean().backward()
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f'batch=4  峰值显存: {mb:.0f} MB  ({mb/1024:.1f} GB)')
    print(f'RTX 3090 有 24 GB，余量: {24 - mb/1024:.1f} GB')
    torch.cuda.empty_cache()
else:
    print('CPU 模式，跳过显存检查')

print('\n✓ Day 3 完成')

batch=4  峰值显存: 3278 MB  (3.2 GB)
RTX 3090 有 24 GB，余量: 20.8 GB

✓ Day 3 完成


---
## 💾 保存到 GitHub

In [22]:
import subprocess, os
from pathlib import Path

REPO_DIR = r'D:\geology AI'
os.chdir(REPO_DIR)

def git(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=REPO_DIR)
    out = (r.stdout + r.stderr).strip()
    if out: print(out)

git('git add notebooks/07_lora_model.ipynb')
git('git commit -m "Week2 Day3: PrithviMAE + LoRA(r=8) on encoder.blocks.attn + SegDecoder verified"')
git('git push origin main')
print('\n✓ 已推送到 GitHub')

[main 1dd7432] Week2 Day3: PrithviMAE + LoRA(r=8) on encoder.blocks.attn + SegDecoder verified
 1 file changed, 310 insertions(+)
 create mode 100644 notebooks/07_lora_model.ipynb
To https://github.com/lofophil/geo-foundation-experiments.git
   8ad649a..1dd7432  main -> main

✓ 已推送到 GitHub
